# Travel Agent with Redis Cloud, Snowflake Cortex, Free Web Search & Weather API

- Snowflake Cortex for LLM (chat) & embeddings (arctic‑embed‑l‑v2.0)
- Redis Cloud for long‑term vector memory + checkpoints + transcripts
- DuckDuckGo web search (free, no key)
- OpenWeatherMap API for accurate weather (free tier, 1,000 calls/day)
- Date utility (no external call)
- Tool‑call cap (3 per turn) and robust JSON parsing
- Conversation selector on startup
- In‑chat commands: `exit`, `debug`, `history`, `memories`

# Cell 1 – Install dependencies

In [16]:
%pip install langgraph langgraph-checkpoint redis redisvl ulid pydantic requests python-dotenv duckduckgo-search beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


# Cell 2 – Environment variables

In [17]:
import os, getpass
from dotenv import load_dotenv
load_dotenv()

def _set_env(key: str):
    if key not in os.environ:
        os.environ[key] = getpass.getpass(f"{key}: ")

_set_env("SNOWFLAKE_ACCOUNT")
_set_env("SNOWFLAKE_PAT")
_set_env("MODEL")
_set_env("REDIS_URL")
_set_env("OPENWEATHER_API_KEY")   # get free key from openweathermap.org

# Cell 3 – Connect to Redis

In [18]:
from redis import Redis
REDIS_URL = os.environ["REDIS_URL"]
redis_client = Redis.from_url(REDIS_URL)
redis_client.ping()
print("✅ Connected to Redis")

✅ Connected to Redis


# Cell 4 – Pydantic models

In [19]:
import ulid
from datetime import datetime
from enum import Enum
from typing import List, Optional
from pydantic import BaseModel, Field

class MemoryType(str, Enum):
    EPISODIC = "episodic"
    SEMANTIC = "semantic"

class Memory(BaseModel):
    content: str
    memory_type: MemoryType
    metadata: str

class StoredMemory(Memory):
    id: str
    memory_id: ulid.ULID = Field(default_factory=lambda: ulid.ULID())
    created_at: datetime = Field(default_factory=datetime.now)
    user_id: Optional[str] = None
    thread_id: Optional[str] = None
    memory_type: Optional[MemoryType] = None
print("✅ Models ready")

✅ Models ready


# Cell 5 – RedisVL vector index

In [20]:
from redisvl.index import SearchIndex
from redisvl.schema.schema import IndexSchema

VECTOR_DIM = 1024

memory_schema = IndexSchema.from_dict({
    "index": {"name": "agent_memories", "prefix": "memory", "key_separator": ":", "storage_type": "json"},
    "fields": [
        {"name": "content", "type": "text"},
        {"name": "memory_type", "type": "tag"},
        {"name": "metadata", "type": "text"},
        {"name": "created_at", "type": "text"},
        {"name": "user_id", "type": "tag"},
        {"name": "memory_id", "type": "tag"},
        {"name": "thread_id", "type": "tag"},
        {"name": "embedding", "type": "vector", "attrs": {"algorithm": "flat", "dims": VECTOR_DIM, "distance_metric": "cosine", "datatype": "float32"}},
    ],
})

long_term_memory_index = SearchIndex(schema=memory_schema, redis_client=redis_client, validate_on_load=True)
long_term_memory_index.create(overwrite=True)
print("✅ Long‑term memory index ready")

✅ Long‑term memory index ready


# Cell 6 – Snowflake embedding helper

In [21]:
import requests, json
ACCOUNT = os.environ["SNOWFLAKE_ACCOUNT"]
PAT = os.environ["SNOWFLAKE_PAT"]
EMBED_MODEL = "snowflake-arctic-embed-l-v2.0"

def embed_text(text: str) -> List[float]:
    url = f"https://{ACCOUNT}.snowflakecomputing.com/api/v2/cortex/inference:embed"
    headers = {"Authorization": f"Bearer {PAT}", "Content-Type": "application/json"}
    payload = {"model": EMBED_MODEL, "input": text}
    response = requests.post(url, headers=headers, json=payload)
    if response.status_code != 200:
        raise RuntimeError(f"Embed API error {response.status_code}: {response.text}")
    return response.json()["data"][0]["embedding"]

# Cell 7 – Memory operations (store, retrieve, deduplicate)

In [22]:
import logging
from redisvl.query import VectorRangeQuery
from redisvl.query.filter import Tag

logger = logging.getLogger(__name__)
SYSTEM_USER_ID = "system"

def similar_memory_exists(
    content: str,
    memory_type: MemoryType,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
    distance_threshold: float = 0.1,
) -> bool:
    embedding = embed_text(content)
    filters = (Tag("user_id") == user_id) & (Tag("memory_type") == memory_type.value)
    if thread_id:
        filters = filters & (Tag("thread_id") == thread_id)
    q = VectorRangeQuery(
        vector=embedding,
        num_results=1,
        vector_field_name="embedding",
        filter_expression=filters,
        distance_threshold=distance_threshold,
        return_fields=["id"],
    )
    return len(long_term_memory_index.query(q)) > 0

def store_memory(
    content: str,
    memory_type: MemoryType,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
    metadata: Optional[str] = None,
) -> None:
    if metadata is None:
        metadata = "{}"
    if similar_memory_exists(content, memory_type, user_id, thread_id):
        logger.info("Similar memory exists — skipping.")
        return
    embedding = embed_text(content)
    memory_data = {
        "user_id": user_id or SYSTEM_USER_ID,
        "content": content,
        "memory_type": memory_type.value,
        "metadata": metadata,
        "created_at": datetime.now().isoformat(),
        "embedding": embedding,
        "memory_id": str(ulid.ULID()),
        "thread_id": thread_id or "",
    }
    long_term_memory_index.load([memory_data])
    logger.info(f"💾 Stored [{memory_type.value}] memory: {content!r}")

def retrieve_memories(
    query: str,
    memory_type: Optional[MemoryType] = None,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
    distance_threshold: float = 0.3,
    limit: int = 5,
) -> List[StoredMemory]:
    embedding = embed_text(query)
    filters = [f"@user_id:{{{user_id or SYSTEM_USER_ID}}}"]
    if memory_type:
        filters.append(f"@memory_type:{{{memory_type.value}}}")
    if thread_id:
        filters.append(f"@thread_id:{{{thread_id}}}")
    q = VectorRangeQuery(
        vector=embedding,
        return_fields=["content", "memory_type", "metadata", "created_at", "memory_id", "thread_id", "user_id"],
        num_results=limit,
        vector_field_name="embedding",
        distance_threshold=distance_threshold,
        dialect=2,
    )
    q.set_filter(" ".join(filters))
    results = long_term_memory_index.query(q)
    memories = []
    for doc in results:
        try:
            memories.append(StoredMemory(
                id=doc["id"],
                memory_id=doc["memory_id"],
                user_id=doc["user_id"],
                thread_id=doc.get("thread_id") or None,
                memory_type=MemoryType(doc["memory_type"]),
                content=doc["content"],
                created_at=doc["created_at"],
                metadata=doc["metadata"],
            ))
        except Exception as e:
            logger.error(f"Error parsing memory: {e}")
    return memories
print("✅ Memory operations ready")

✅ Memory operations ready


# Cell 8 – Tools (memory, web search, weather, date)

In [23]:
from duckduckgo_search import DDGS
import re, requests, datetime

# ── Memory tools ───────────────────────────────────────────────────────────
def store_memory_tool(
    content: str,
    memory_type: str,
    metadata: Optional[dict] = None,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
) -> str:
    try:
        mem_type = MemoryType(memory_type)
        store_memory(content, mem_type, user_id, thread_id, str(metadata) if metadata else None)
        return f"✅ Stored [{mem_type.value}] memory: {content}"
    except Exception as e:
        return f"❌ Error storing memory: {e}"

def retrieve_memories_tool(
    query: str,
    memory_type: Optional[str] = None,
    limit: int = 5,
    user_id: str = SYSTEM_USER_ID,
    thread_id: Optional[str] = None,
) -> str:
    try:
        mem_type = MemoryType(memory_type) if memory_type else None
        memories = retrieve_memories(query, mem_type, user_id, thread_id, limit=limit)
        if not memories:
            return "No relevant memories found."
        lines = ["🧠 Long-term memories:"]
        for m in memories:
            lines.append(f"  - [{m.memory_type.value}] {m.content}")
        return "\n".join(lines)
    except Exception as e:
        return f"❌ Error retrieving memories: {e}"

# --- Date tool (no external call) ---------------------------------------
def get_current_date_tool(**kwargs) -> str:
    """Return today's date in a human‑readable format."""
    now = datetime.datetime.now()
    return f"Today is {now.strftime('%A, %B %d, %Y')}. Tomorrow will be {(now + datetime.timedelta(days=1)).strftime('%A, %B %d, %Y')}."

# --- Weather tool (OpenWeatherMap) --------------------------------------
def get_weather_tool(city: str, country_code: str = "", units: str = "metric", **kwargs) -> str:
    api_key = os.environ.get("OPENWEATHER_API_KEY")
    if not api_key:
        return "ERROR: OPENWEATHER_API_KEY not set."
    q = f"{city},{country_code}" if country_code else city
    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {"q": q, "appid": api_key, "units": units}
    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        d = r.json()
        temp = d["main"]["temp"]
        feels = d["main"]["feels_like"]
        cond = d["weather"][0]["description"].capitalize()
        hum = d["main"]["humidity"]
        wind = d["wind"]["speed"]
        return (f"Weather in {d['name']}, {d['sys']['country']}:\n"
                f"🌡️ Temp: {temp}°{'C' if units=='metric' else 'F'}\n"
                f"🌡️ Feels: {feels}°{'C' if units=='metric' else 'F'}\n"
                f"☁️ {cond}\n💧 Humidity: {hum}%\n💨 Wind: {wind} m/s")
    except Exception as e:
        return f"Error fetching weather: {e}"

# --- Web search tool (free, DuckDuckGo) ---------------------------------
def web_search_tool(query: str, max_results: int = 5, **kwargs) -> str:
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
        if not results:
            return f"No web results found for '{query}'."
        output = [f"🌐 Web search results for '{query}':"]
        for r in results:
            title = r.get('title', 'No title')
            snippet = r.get('body', '')
            url = r.get('href', '')
            snippet = re.sub(r'\s+', ' ', snippet).strip()
            output.append(f"- **{title}**\n  {snippet}\n  🔗 {url}")
        return "\n\n".join(output)
    except Exception as e:
        return f"Error performing web search: {e}"

# --- Tool registry ------------------------------------------------------
TOOLS = [
    {"name": "store_memory", "func": store_memory_tool,
     "description": "Store a fact (content) with memory_type='episodic' or 'semantic'. Example: {\"tool\": \"store_memory\", \"arguments\": {\"content\": \"Rahmath likes spicy food\", \"memory_type\": \"episodic\"}}"},
    {"name": "retrieve_memories", "func": retrieve_memories_tool,
     "description": "Recall past facts. Example: {\"tool\": \"retrieve_memories\", \"arguments\": {\"query\": \"pin code\", \"memory_type\": \"episodic\", \"limit\": 3}}"},
    {"name": "get_current_date", "func": get_current_date_tool,
     "description": "Get today's date and tomorrow's date. No arguments needed."},
    {"name": "get_weather", "func": get_weather_tool,
     "description": "Get current weather for a city. Parameters: city (str), country_code (optional, e.g., 'IN'), units ('metric' or 'imperial')."},
    {"name": "web_search", "func": web_search_tool,
     "description": "Search the web for general info. Parameters: query (str), max_results (int, default=5)."},
]
print("✅ Tools ready")

✅ Tools ready


# Cell 9 – Snowflake Cortex LLM (improved JSON extraction)

In [24]:
import re, json, ulid, requests, logging
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.callbacks import CallbackManagerForLLMRun
from typing import Any, List, Optional, Sequence

logger = logging.getLogger(__name__)

def extract_tool_call(text: str) -> Optional[dict]:
    """Extract a JSON object with 'tool' and 'arguments' keys, even from malformed text."""
    # Remove markdown code fences
    cleaned = re.sub(r"```(?:json)?\n?|```", "", text).strip()
    # Try to find the outermost JSON object
    start = cleaned.find('{')
    if start == -1:
        return None
    # Count braces to find the matching closing brace
    depth = 0
    for i, ch in enumerate(cleaned[start:], start=start):
        if ch == '{':
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                candidate = cleaned[start:i+1]
                try:
                    obj = json.loads(candidate)
                    if "tool" in obj and "arguments" in obj:
                        return obj
                except json.JSONDecodeError:
                    pass
                break
    return None

class SnowflakeCortexLLM(BaseChatModel):
    account: str
    pat: str
    model: str
    temperature: float = 0.3   # lower for more deterministic JSON

    def _generate(self, messages: List[BaseMessage], stop=None, run_manager=None, **kwargs) -> ChatResult:
        # Convert messages to API format (same as before)
        api_messages = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                api_messages.append({"role": "system", "content": msg.content})
            elif isinstance(msg, HumanMessage):
                api_messages.append({"role": "user", "content": msg.content})
            elif isinstance(msg, AIMessage):
                api_messages.append({"role": "assistant", "content": msg.content})
            elif isinstance(msg, ToolMessage):
                api_messages.append({"role": "user", "content": f"[Tool result for '{msg.name}']:\n{msg.content}"})

        url = f"https://{self.account}.snowflakecomputing.com/api/v2/cortex/v1/chat/completions"
        headers = {"Authorization": f"Bearer {self.pat}", "Content-Type": "application/json"}
        payload = {"model": self.model, "messages": api_messages, "temperature": self.temperature}
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code != 200:
            raise RuntimeError(f"Snowflake API error {response.status_code}: {response.text}")

        data = response.json()
        raw_content = data["choices"][0]["message"]["content"].strip()
        logger.debug(f"Raw LLM: {raw_content!r}")

        # Fallback for utterly malformed responses
        if raw_content in ("}", "", "{}"):
            fallback = "I'm sorry, I had a technical issue. Please rephrase your question."
            # Try a simpler request
            fallback_payload = {"model": self.model, "messages": [{"role": "user", "content": "Reply with a friendly greeting."}], "temperature": 0.3}
            fb = requests.post(url, headers=headers, json=fallback_payload)
            if fb.status_code == 200:
                raw_content = fb.json()["choices"][0]["message"]["content"].strip()
            else:
                raw_content = fallback

        # Extract tool call
        tool_call = extract_tool_call(raw_content)
        clean_content = raw_content
        if tool_call:
            # Remove the JSON block from the content
            clean_content = re.sub(r"\{[\"']tool[\"'][\s\S]*?\}", "", raw_content).strip()
            if not clean_content:
                clean_content = "Using a tool to help you."

        ai_msg = AIMessage(
            content=clean_content,
            tool_calls=[{
                "name": tool_call["tool"],
                "args": tool_call["arguments"],
                "id": f"call_{ulid.ULID()}",
            }] if tool_call else []
        )
        return ChatResult(generations=[ChatGeneration(message=ai_msg)])

    @property
    def _llm_type(self) -> str:
        return "snowflake-cortex"

# Initialize
account = os.environ["SNOWFLAKE_ACCOUNT"]
pat = os.environ["SNOWFLAKE_PAT"]
model = os.environ["MODEL"]
llm = SnowflakeCortexLLM(account=account, pat=pat, model=model, temperature=0.0)
print("✅ Snowflake Cortex LLM ready")

✅ Snowflake Cortex LLM ready


# Cell 10 – Conversation persistence (transcripts + selector)

In [25]:
from datetime import datetime
import json
import logging

CONV_PREFIX = "conversation"

def save_transcript(state, thread_id: str, user_id: str) -> None:
    transcript = []
    for m in state["messages"]:
        if isinstance(m, HumanMessage):
            transcript.append({"role": "user", "content": m.content})
        elif isinstance(m, AIMessage):
            content = m.content.strip() if m.content else ""
            if content and not extract_tool_call(content):
                transcript.append({"role": "assistant", "content": content})
        elif isinstance(m, ToolMessage):
            transcript.append({"role": "tool", "name": m.name, "content": m.content})
        elif isinstance(m, SystemMessage):
            transcript.append({"role": "summary", "content": m.content})
    key = f"{CONV_PREFIX}:{user_id}:{thread_id}"
    redis_client.set(key, json.dumps(transcript))
    logger.info(f"💾 Transcript saved → {key} ({len(transcript)} turns)")

def load_transcript(thread_id: str, user_id: str) -> list:
    key = f"{CONV_PREFIX}:{user_id}:{thread_id}"
    raw = redis_client.get(key)
    return json.loads(raw) if raw else []

def get_all_transcript_keys(user_id: str) -> list:
    pattern = f"{CONV_PREFIX}:{user_id}:*"
    return sorted([k.decode() for k in redis_client.keys(pattern)])

def select_conversation(user_id: str = "demo_user") -> tuple:
    """Interactive selector; returns (thread_id, transcript)."""
    keys = get_all_transcript_keys(user_id)
    print("\n" + "=" * 55)
    print("  📚 CONVERSATION SELECTOR")
    print("=" * 55)
    if not keys:
        print("  No previous conversations found. Starting new.\n")
        thread_id = f"thread_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        return thread_id, []
    print("  [0] 🆕 Start a new conversation\n")
    for i, key in enumerate(keys, start=1):
        thread_part = key.split(":", 2)[2] if key.count(":") >= 2 else key
        raw = redis_client.get(key)
        turns = len(json.loads(raw)) if raw else 0
        preview = ""
        if raw:
            transcript = json.loads(raw)
            for turn in transcript:
                if turn.get("role") == "user":
                    preview = turn.get("content", "")[:60]
                    break
        print(f"  [{i}] 🗂  {thread_part}")
        print(f"       {turns} turns  |  \"{preview}{'...' if len(preview) == 60 else ''}\"")
        print()
    print("=" * 55)
    while True:
        try:
            choice = input(f"  Choose [0-{len(keys)}]: ").strip()
        except (EOFError, KeyboardInterrupt):
            choice = "0"
        if choice == "0":
            thread_id = f"thread_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
            print(f"\n  🆕 New conversation started: {thread_id}\n")
            return thread_id, []
        if choice.isdigit() and 1 <= int(choice) <= len(keys):
            idx = int(choice) - 1
            chosen_key = keys[idx]
            thread_id = chosen_key.split(":", 2)[2]
            transcript = json.loads(redis_client.get(chosen_key))
            print(f"\n  📂 Resuming: {thread_id} ({len(transcript)} turns)\n")
            return thread_id, transcript
        print(f"  ❌ Invalid choice. Enter a number between 0 and {len(keys)}.")

def print_transcript(thread_id: str, user_id: str = "demo_user") -> None:
    transcript = load_transcript(thread_id, user_id)
    if not transcript:
        print(f"No transcript found for user={user_id!r} thread={thread_id!r}")
        return
    print(f"\n{'='*60}")
    print(f"TRANSCRIPT  user={user_id}  thread={thread_id}")
    print(f"{'='*60}")
    for turn in transcript:
        role = turn["role"].upper()
        name = f" ({turn['name']})" if turn.get("name") else ""
        print(f"\n[{role}{name}]\n{turn.get('content', '')}")
    print()

print("✅ Conversation transcript helpers ready")

✅ Conversation transcript helpers ready


# Cell 11 – LangGraph workflow with updated prompt

In [32]:
from langgraph.graph import StateGraph, END
from langgraph.graph.message import MessagesState
from langgraph.checkpoint.redis import RedisSaver
from langchain_core.runnables.config import RunnableConfig
import re
from datetime import datetime, timedelta

class RuntimeState(MessagesState):
    tool_call_count: int

MAX_TOOL_CALLS_PER_TURN = 3

SYSTEM_PROMPT = """You are a travel assistant with tools.

Available tools:
1. store_memory(content, memory_type) – content is the fact, memory_type is 'episodic'.
2. retrieve_memories(query) – recall facts.
3. get_current_date() – no args.
4. get_weather(city, country_code, units) – country_code optional, units 'metric' or 'imperial'.
5. web_search(query) – general web search.

To call a tool, respond with ONLY this JSON:
{"tool": "tool_name", "arguments": {"arg1": "value"}}

If you don't need a tool, reply in plain text.

Now answer the user.
"""

# --- Agent node with retry ---
def respond_to_user(state: RuntimeState, config: RunnableConfig) -> RuntimeState:
    user_msgs = [m for m in state["messages"] if isinstance(m, HumanMessage)]
    if not user_msgs:
        return state

    user_input = user_msgs[-1].content.strip()
    configurable = config.get("configurable", {}) if config else {}
    user_id = configurable.get("user_id", SYSTEM_USER_ID)
    thread_id = configurable.get("thread_id")

    # ------------------------------------------------------------
    # 1. RULE-BASED HANDLING FOR COMMON TASKS
    # ------------------------------------------------------------
    
    # --- Store memory: name ---
    store_name = re.search(r"(?:my name is|i am|call me)\s+(\w+)", user_input, re.IGNORECASE)
    if store_name:
        name = store_name.group(1)
        store_memory_tool(content=f"User's name is {name}", memory_type="episodic", user_id=user_id, thread_id=thread_id)
        state["messages"].append(AIMessage(content=f"✅ Stored your name as {name}."))
        return state

    # --- Store memory: pin code ---
    pin_match = re.search(r"(?:pin code|pincode|postal code)\s*(?:is|of)\s*(\w+)?\s*(\d{6})", user_input, re.IGNORECASE)
    if pin_match:
        pin = pin_match.group(2) if pin_match.group(2) else pin_match.group(1)
        store_memory_tool(content=f"Hampi pin code is {pin}", memory_type="episodic", user_id=user_id, thread_id=thread_id)
        state["messages"].append(AIMessage(content=f"✅ Stored Hampi pin code as {pin}."))
        return state

    # --- Retrieve memory: name ---
    if re.search(r"(?:what is|tell me|do you know)\s*(?:my name|name\?)", user_input, re.IGNORECASE):
        memories = retrieve_memories(query="name", memory_type=MemoryType.EPISODIC, user_id=user_id, thread_id=thread_id, limit=1)
        if memories:
            answer = f"Your name is {memories[0].content}."
        else:
            answer = "I don't know your name yet. Please tell me."
        state["messages"].append(AIMessage(content=answer))
        return state

    # --- Retrieve memory: pin code ---
    if re.search(r"(?:pin code|pincode|postal code)\s*(?:of hampi|for hampi|hampi\s*pin)", user_input, re.IGNORECASE):
        memories = retrieve_memories(query="Hampi pin code", memory_type=MemoryType.EPISODIC, user_id=user_id, thread_id=thread_id, limit=1)
        if memories:
            answer = f"The pin code of Hampi is {memories[0].content}."
        else:
            answer = "I don't have the pin code for Hampi stored. Please tell me."
        state["messages"].append(AIMessage(content=answer))
        return state

    # --- Date ---
    if re.search(r"(?:today|tomorrow|date|day)", user_input, re.IGNORECASE):
        now = datetime.now()
        tomorrow = now + timedelta(days=1)
        answer = f"Today is {now.strftime('%A, %B %d, %Y')}. Tomorrow is {tomorrow.strftime('%A, %B %d, %Y')}."
        state["messages"].append(AIMessage(content=answer))
        return state

    # --- Weather ---
    weather_match = re.search(r"weather\s+(?:in|for)\s*(\w+)(?:\s*,\s*(\w{2}))?", user_input, re.IGNORECASE)
    if weather_match:
        city = weather_match.group(1)
        country = weather_match.group(2) if weather_match.group(2) else ""
        result = get_weather_tool(city=city, country_code=country, units="metric")
        state["messages"].append(AIMessage(content=result))
        return state

    # --- Web search for news / general info (explicit) ---
    search_match = re.search(r"(?:news|search|find|look up)\s+(?:about\s+)?(.+)", user_input, re.IGNORECASE)
    if search_match:
        query = search_match.group(1).strip()
        if query:
            result = web_search_tool(query=query, max_results=5)
            state["messages"].append(AIMessage(content=result))
            return state

    # ------------------------------------------------------------
    # 2. FALLBACK: USE LLM FOR GENERAL CONVERSATION
    # ------------------------------------------------------------
    tool_calls_this_turn = state.get("tool_call_count", 0)
    llm_messages = [SystemMessage(content=SYSTEM_PROMPT)]
    for m in state["messages"]:
        if isinstance(m, HumanMessage):
            llm_messages.append(HumanMessage(content=m.content))
        elif isinstance(m, AIMessage):
            if m.content and not extract_tool_call(m.content):
                llm_messages.append(AIMessage(content=m.content))
        elif isinstance(m, ToolMessage):
            llm_messages.append(HumanMessage(content=f"[Tool result for '{m.name}']:\n{m.content}"))

    if tool_calls_this_turn >= MAX_TOOL_CALLS_PER_TURN:
        llm_messages.append(HumanMessage(
            content="You have used the maximum tool calls. Reply directly in plain text."
        ))

    ai_msg = llm.invoke(llm_messages)

    # --- If LLM response is malformed, fallback to web search ---
    if not ai_msg.content or ai_msg.content.strip() in ("}", ""):
        logger.warning("Malformed LLM response – falling back to web search on user query.")
        # Try a web search with the full user input
        search_result = web_search_tool(query=user_input, max_results=5)
        state["messages"].append(AIMessage(content=search_result))
        return state

    state["messages"].append(ai_msg)
    return state

# --- execute_tools node (unchanged) ---
def execute_tools(state: RuntimeState, config: RunnableConfig) -> RuntimeState:
    ai_msgs = [m for m in state["messages"] if isinstance(m, AIMessage) and getattr(m, "tool_calls", None)]
    if not ai_msgs:
        return state

    latest = ai_msgs[-1]
    configurable = config.get("configurable", {}) if config else {}
    user_id = configurable.get("user_id", SYSTEM_USER_ID)
    thread_id = configurable.get("thread_id")

    for tc in latest.tool_calls:
        tool_name = tc["name"]
        tool_args = dict(tc["args"])
        tool_func = next((t["func"] for t in TOOLS if t["name"] == tool_name), None)

        if not tool_func:
            result = f"Tool '{tool_name}' not found."
        else:
            tool_args.setdefault("user_id", user_id)
            tool_args.setdefault("thread_id", thread_id)
            try:
                result = tool_func(**tool_args)
                logger.info(f"Tool '{tool_name}' result: {result!r}")
            except Exception as e:
                result = f"Error executing tool '{tool_name}': {e}"
                logger.error(result)

        state["messages"].append(ToolMessage(content=str(result), tool_call_id=tc["id"], name=tool_name))

    state["tool_call_count"] = state.get("tool_call_count", 0) + 1
    return state

# --- summarize node (unchanged) ---
MESSAGE_THRESHOLD = 8
def summarize_conversation(state: RuntimeState, config: RunnableConfig) -> RuntimeState:
    messages = state["messages"]
    if len(messages) < MESSAGE_THRESHOLD:
        return state

    readable = []
    for m in messages:
        if isinstance(m, HumanMessage):
            readable.append(f"User: {m.content}")
        elif isinstance(m, AIMessage):
            content = m.content.strip() if m.content else ""
            if content and not extract_tool_call(content):
                readable.append(f"Assistant: {content}")

    if not readable:
        return state

    summary_prompt = (
        "Summarise this travel assistant conversation in 3-5 sentences. "
        "Focus on destinations, user preferences, and decisions made.\n\n"
        + "\n".join(readable)
    )
    summary_msg = llm.invoke([
        SystemMessage(content="You are a concise conversation summariser."),
        HumanMessage(content=summary_prompt)
    ])
    logger.info(f"Summarised {len(messages)} messages.")

    summary_sys = SystemMessage(content=f"Summary of conversation so far:\n\n{summary_msg.content}\n\nContinue from here.")
    keep = messages[-2:] if len(messages) >= 2 else messages
    state["messages"] = [summary_sys] + keep
    return state

# --- routing ---
def decide_next(state: RuntimeState) -> str:
    last = state["messages"][-1] if state["messages"] else None
    if state.get("tool_call_count", 0) >= MAX_TOOL_CALLS_PER_TURN:
        return "summarize"
    if isinstance(last, AIMessage) and getattr(last, "tool_calls", None):
        return "execute_tools"
    return "summarize"

# --- compile graph ---
workflow = StateGraph(RuntimeState)
workflow.add_node("agent", respond_to_user)
workflow.add_node("execute_tools", execute_tools)
workflow.add_node("summarize", summarize_conversation)

workflow.set_entry_point("agent")
workflow.add_conditional_edges(
    "agent",
    decide_next,
    {"execute_tools": "execute_tools", "summarize": "summarize"},
)
workflow.add_edge("execute_tools", "agent")
workflow.add_edge("summarize", END)

redis_saver = RedisSaver(redis_client=redis_client)
redis_saver.setup()
graph = workflow.compile(checkpointer=redis_saver)

print("✅ Graph compiled with retry logic")

✅ Graph compiled with retry logic


# Cell 12 – Interactive loop (with selector and commands)

In [33]:
from langchain_core.messages import HumanMessage

def main(user_id: str = "demo_user", verbose: bool = False):

    # ── Conversation selector ──────────────────────────────────────────────
    thread_id, _ = select_conversation(user_id=user_id)   # transcript not used for state restore

    print(f"🌍 Travel Assistant with Memory (Snowflake Cortex + Redis checkpoints)")
    print(f"   user={user_id}  thread={thread_id}")
    print("   Commands: exit | quit | debug | history | memories\n")

    config = {"configurable": {"thread_id": thread_id, "user_id": user_id}, "recursion_limit": 50}

    # The graph will automatically load the saved state from RedisSaver for this thread_id.
    # We start with an empty state (if new thread, it's empty).
    state = RuntimeState(messages=[], tool_call_count=0)

    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            # Save a final transcript (optional)
            save_transcript(state, thread_id, user_id)
            break

        if not user_input:
            continue

        if user_input.lower() in ("exit", "quit"):
            print("Goodbye!")
            save_transcript(state, thread_id, user_id)
            break

        if user_input.lower() == "debug":
            verbose = not verbose
            print(f"  [verbose mode {'ON' if verbose else 'OFF'}]\n")
            continue

        if user_input.lower() == "history":
            print_transcript(thread_id, user_id)
            continue

        if user_input.lower() == "memories":
            keys = redis_client.keys("memory:*")
            print(f"\n🧠 Long-term memories ({len(keys)} total):")
            for key in sorted(keys):
                data = redis_client.json().get(key)
                if data:
                    ts  = data.get("created_at", "")[:19]
                    mt  = data.get("memory_type", "?")
                    cnt = data.get("content", "")
                    print(f"  [{mt:8s}] {ts}  {cnt}")
            print()
            continue

        # ── Reset tool‑call counter for new user turn ──────────────────────
        state["tool_call_count"] = 0
        state["messages"].append(HumanMessage(content=user_input))

        try:
            # The graph.stream will load the checkpointed state automatically.
            # We pass the config with thread_id, and the checkpointer will merge.
            for result in graph.stream(state, config=config, stream_mode="values"):
                state = RuntimeState(**result)

            # Find the last clean assistant reply
            reply = None
            for m in reversed(state["messages"]):
                if isinstance(m, AIMessage):
                    content = m.content.strip() if m.content else ""
                    if content and not extract_tool_call(content):
                        reply = content
                        break

            if reply:
                print(f"\nAssistant: {reply}\n")
            else:
                print("\nAssistant: (no reply generated — try again)\n")

            # Save transcript after each turn (for history view)
            save_transcript(state, thread_id, user_id)

            if verbose:
                print(f"── DEBUG (tool_call_count={state.get('tool_call_count', 0)}) ──")
                for i, m in enumerate(state["messages"]):
                    kind = type(m).__name__
                    content = (m.content or "")[:120].replace("\n", " ")
                    tc = f" [calls={[t['name'] for t in getattr(m, 'tool_calls', [])]}]" if getattr(m, "tool_calls", None) else ""
                    print(f"  [{i:02d}] {kind}{tc}: {content!r}")
                print("──────────────────────────────────────\n")

        except Exception as e:
            logger.error(f"Error during graph execution: {e}", exc_info=True)
            print(f"\n⚠️  Error: {e}\n")

    return state

if __name__ == "__main__":
    user_id = input("Enter user ID (default demo_user): ") or "demo_user"
    final_state = main(user_id=user_id, verbose=False)


  📚 CONVERSATION SELECTOR
  [0] 🆕 Start a new conversation

  [1] 🗂  thread_20260806_131441
       6 turns  |  "hello"

  [2] 🗂  thread_20260806_131846
       18 turns  |  "hi"

  [3] 🗂  thread_20260806_134036
       61 turns  |  "do the websearch for the rains on 7 august 2026 weather fore..."

  [4] 🗂  thread_20260806_153647
       13 turns  |  "whats my name"

  [5] 🗂  thread_20260806_154147
       14 turns  |  "name is rahmath hampi code is 583239 remember this. also ham..."

  [6] 🗂  thread_20260806_154345
       4 turns  |  "name?"

  [7] 🗂  thread_20260806_154501
       20 turns  |  "name is rahmath and hampi code is 548574 and tomorrows date?..."

  [8] 🗂  thread_20260806_154651
       4 turns  |  "name?"

  [9] 🗂  thread_20260806_202625
       8 turns  |  "weather of hampi"

  [10] 🗂  thread_20260807_051725
       21 turns  |  "whats my name?"

  [11] 🗂  thread_20260807_054921
       15 turns  |  "whats my name"


  🆕 New conversation started: thread_20260807_055429

🌍 Travel

C:\Users\rahma\AppData\Local\Temp\ipykernel_36724\4203407234.py:71: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Assistant: 🌐 Web search results for 'on hampi':

- **Wakiya一笑美茶樓｜レストラン｜脇屋友詞の中国料理 Wakiya ...**
  Wakiyaのフカヒレが特別な理由 一年に一度だけ！ フカヒレ雄雌食べくらべ 毎年6月に開催するWakiya一笑美茶樓の「フカヒレ雄雌食べくらべの会」 スペシャリテであるフカ...
  🔗 http://www.wakiya.co.jp/restaurants/ichiemi/

- **脇屋友詞の中国料理 Wakiya一笑美茶樓**
  脇屋友詞の中国料理 Wakiya一笑美茶樓 ようこそ中国茶の世界へ 僕が初めて中国茶に魅せられたのは35年ほど前、本場台湾で烏龍茶を飲んだ時のことで...
  🔗 https://www.wakiya.co.jp/

- **Wakiya一笑美茶樓｜レストラン｜脇屋友詞の中国料理 Wakiya ...**
  脇屋友詞の中国料理 Wakiya一笑美茶樓 アフタヌーン・ティー 4,800円 点心3種、中華粥、デザート、中国菓子と中国茶8種フリーフローのセット 6,000円 4,800円のセットにグラスのスパークリングワイン1杯付 …
  🔗 https://www.wakiya.co.jp/restaurants/ichiemi/menu/

- **Turandot 臥龍居｜レストラン｜脇屋友詞の中国料理 Wakiya ...**
  脇屋友詞の中国料理 Wakiya一笑美茶樓 最新情報一覧へ 〒107-0052 東京都港区赤坂6-16-10 Y's CROSS ROAD1,2F TEL 03-3568-7190 電車でお越しの際は 赤坂駅（東京メトロ千代田線）・7番出口より徒歩6分 …
  🔗 https://www.wakiya.co.jp/restaurants/akasaka/

- **Wakiya迎賓茶樓｜レストラン｜脇屋友詞の中国料理 Wakiya ...**
  クリスマスはWakiyaで和やかに Wakiyaのクリスマススペシャルメニュー 大切な方と美味しい料理でとっておきの夜を。 Wakiya一笑美茶樓、トゥーランドット臥龍居では1...
  🔗 https://www.wakiya.co.jp/restaurants/

Malformed LLM response – falling back to web search on user query.
C:\Users\rahma\AppData\Local\Temp\ipykernel_36724\4203407234.py:71: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Assistant: No web results found for 'tell me about pre historic  drawings in caves in hampi'.



Malformed LLM response – falling back to web search on user query.
C:\Users\rahma\AppData\Local\Temp\ipykernel_36724\4203407234.py:71: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Assistant: No web results found for 'tell me about hampi and places to visit'.

Goodbye!


# Cell 13 – Inspect conversations (optional)

In [28]:
# List all transcripts
keys = redis_client.keys("conversation:demo_user:*")
print("Transcripts:")
for k in sorted(keys):
    print(k.decode())

Transcripts:
conversation:demo_user:thread_20260806_131441
conversation:demo_user:thread_20260806_131846
conversation:demo_user:thread_20260806_134036
conversation:demo_user:thread_20260806_153647
conversation:demo_user:thread_20260806_154147
conversation:demo_user:thread_20260806_154345
conversation:demo_user:thread_20260806_154501
conversation:demo_user:thread_20260806_154651
conversation:demo_user:thread_20260806_202625
conversation:demo_user:thread_20260807_051725
conversation:demo_user:thread_20260807_054921


In [31]:
# View a specific transcript
print_transcript("thread_20260807_054921", "demo_user")


TRANSCRIPT  user=demo_user  thread=thread_20260807_054921

[USER]
whats my name

[ASSISTANT]
I don't have any information about your name. This is the beginning of our conversation, and I don't have any prior knowledge about you. If you'd like to share your name, I can store it in my memory for our conversation.

[USER]
My name is Rahmath

[ASSISTANT]
✅ Stored your name as Rahmath.

[USER]
Hampi pin code is 583239

[ASSISTANT]
✅ Stored Hampi pin code as 583239.

[USER]
What is tomorrow's date?

[ASSISTANT]
Today is Friday, August 07, 2026. Tomorrow is Saturday, August 08, 2026.

[SUMMARY]
Summary of conversation so far:

Here is a summary of the conversation:

Rahmath, the user, shared their name and the pin code of Hampi (583239) with the travel assistant. No specific travel plans or destinations were discussed, but Hampi was mentioned. The conversation also established the current date as August 07, 2026, and the next day's date as August 08, 2026. No travel decisions or preferences

In [30]:
# List all long‑term memories (without embedding)
keys = redis_client.keys("memory:*")
print(f"Total memories: {len(keys)}")
for key in sorted(keys):
    data = redis_client.json().get(key)
    if data:
        print(json.dumps({k: v for k, v in data.items() if k != "embedding"}, indent=2))
        print("-" * 40)

Total memories: 0
